# Nova Visualização — Bússola de Vocabulário dos Planos Nacionais de IA

## Escopo e propósito

Este notebook é uma **análise nova e autocontida**, que não segue o pipeline metodológico
das Skills 02_Análise_Vocab_A/B nem reaproveita nenhum código, corte de vocabulário ou
decisão de normalização já registrados em `analise_geral.ipynb` ou nos registros
individuais de cada país. Todo o pré-processamento de texto (tokenização, remoção de
palavras funcionais, tratamento de bigramas, exclusão de autoidentificadores) é
**refeito do zero**, com uma metodologia própria, documentada célula a célula abaixo.

**Pergunta de pesquisa.** Em vez de medir similaridade par a par entre documentos (como em
`analise_geral.ipynb`), este notebook parte de uma pergunta diferente: existe um
**vocabulário consensual** — o vocabulário-padrão que a maioria dos seis planos nacionais
de IA compartilha — e, além dele, **até quatro polos de vocabulário distintivo**
("Vocabulário A", "B", "C" e "D"), cada um definido estatisticamente a partir dos termos
que mais afastam os documentos uns dos outros? E, para cada documento, qual a proporção
(relativa, não absoluta) do seu vocabulário que pertence a cada uma dessas cinco
categorias?

## Documentos utilizados

Os seis planos nacionais de IA já extraídos (Skill01) nas pastas de país/bloco desta
pesquisa, cada um em `texto_completo`:

| Documento | País/bloco | Arquivo |
|---|---|---|
| PBIA | Brasil | `Brasil/pbia.json` |
| "AI+" Initiative | China | `China/ai_plus.json` |
| New Generation AI Development Plan | China | `China/new_generation_ai_development_plan.json` |
| America's AI Action Plan | Estados Unidos | `Estados Unidos/americas_ai_action_plan.json` |
| Apply AI Strategy | Europa (UE) | `Europa/apply_ai_strategy.json` |
| AI Continent Action Plan | Europa (UE) | `Europa/ai_continent_action_plan.json` |

**A Índia foi excluída desta análise.** A pasta `Índia/` contém apenas um notebook
(`national_plan.ipynb`); não existe, até o momento, nenhum arquivo `.json` extraído pela
Skill01 para o plano nacional indiano. Isso é uma limitação de disponibilidade de dados,
não uma escolha metodológica — a análise usa **todos** os documentos `.json` atualmente
extraídos, que são os seis listados acima. Os campos usados de cada JSON são,
exclusivamente, `titulo`, `pais_ou_bloco` e `texto_completo`, seguindo o mesmo protocolo
de uso do JSON já estabelecido para este projeto (`elementos_descartados`,
`data_extracao`, `data_publicacao` e `fonte` não são utilizados).

## Por que uma comparação relativa é obrigatória

Os seis documentos têm tamanhos brutalmente diferentes — o maior (PBIA) tem quase **6
vezes** mais palavras que o menor ("AI+" Initiative). Qualquer contagem absoluta de
termos favoreceria mecanicamente os documentos mais longos. Por isso, **toda** medida
usada neste notebook — a definição de vocabulário consensual, a posição de cada
documento na bússola, a composição percentual por categoria — é calculada sobre
**frequência relativa** (proporção do vocabulário do próprio documento), nunca sobre
contagem bruta, e cada documento entra nos cálculos que definem os polos com **peso
igual** (1/6), independentemente do seu tamanho. Isso é detalhado na Seção 3, abaixo.

In [ ]:
import json, re, math
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 60)
pd.set_option("display.width", 120)

# paleta categórica (dataviz skill — reference palette, modo claro)
COL_BLUE, COL_ORANGE, COL_AQUA, COL_YELLOW, COL_RED = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e34948"
INK_PRIMARY, INK_SECONDARY, INK_MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, AXIS, SURFACE, POLE_BG = "#e1e0d9", "#c3c2b7", "#fcfcfb", "#232042"

plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["axes.edgecolor"] = AXIS
print("Ambiente pronto.")

## 0. Carregamento dos seis documentos

Caminhos relativos a esta pasta (`Análise Conjunta/`), apontando para os `.json` de cada
país/bloco. Apenas os três campos autorizados pelo protocolo do projeto são lidos.

In [ ]:
FILES = {
    "pbia":           ("../Brasil/pbia.json", "Brasil", "PBIA"),
    "ai_plus":        ("../China/ai_plus.json", "China", '"AI+" Initiative'),
    "new_generation": ("../China/new_generation_ai_development_plan.json", "China", "New Generation AI Dev. Plan"),
    "americas":       ("../Estados Unidos/americas_ai_action_plan.json", "Estados Unidos", "America's AI Action Plan"),
    "apply_ai":       ("../Europa/apply_ai_strategy.json", "Europa (UE)", "Apply AI Strategy"),
    "ai_continent":   ("../Europa/ai_continent_action_plan.json", "Europa (UE)", "AI Continent Action Plan"),
}

docs = {}
for key, (path, bloc, label) in FILES.items():
    with open(path, encoding="utf-8") as f:
        d = json.load(f)
    docs[key] = {"titulo": d["titulo"], "pais_ou_bloco": bloc, "label": label, "texto": d["texto_completo"]}

doc_keys = list(docs.keys())

size_df = pd.DataFrame({
    "documento": [docs[k]["label"] for k in doc_keys],
    "país/bloco": [docs[k]["pais_ou_bloco"] for k in doc_keys],
    "caracteres": [len(docs[k]["texto"]) for k in doc_keys],
    "palavras (aprox.)": [len(docs[k]["texto"].split()) for k in doc_keys],
})
size_df["razão vs. menor documento"] = (size_df["palavras (aprox.)"] / size_df["palavras (aprox.)"].min()).round(1)
size_df

A tabela confirma o problema anunciado na introdução: o PBIA tem cerca de **6,1 vezes**
mais palavras que a "AI+" Initiative. Nenhuma comparação por contagem bruta entre esses
seis documentos seria válida — daí a exigência de normalização relativa em todas as
etapas seguintes.

## 1. Tokenização e normalização — metodologia

Esta é uma tokenização construída para este notebook, independente da Skill
02_Análise_Vocab_A. As decisões, em ordem de aplicação:

1. **Normalização de caracteres.** Aspas e apóstrofos tipográficos (`’ ‘ “ ”`), comuns em
   texto extraído da web (72 ocorrências no `new_generation`, 68 no `americas`), são
   convertidos para as formas ASCII retas antes de qualquer tokenização — sem isso, por
   exemplo, `China's` tokenizaria como dois termos (`china`, `s`) em vez de um.
2. **Proteção de termos ambíguos ou destruídos pela tokenização simples**, antes da
   tokenização geral:
   - `AI+` / `"Artificial Intelligence+"` → rótulo próprio `AI+`. Sem essa proteção, o
     `+` seria descartado e as 9 ocorrências de `AI+` (o nome da própria iniciativa
     chinesa) se confundiriam com as 90 ocorrências do termo genérico `AI` no mesmo
     documento — exatamente o tipo de ambiguidade contextual que a Skill
     02_Análise_Vocab_A já havia identificado como risco de imprecisão grave.
   - `R&D` / `research and development` → rótulo único `R&D` (verificado: 36 ocorrências
     de `R&D`/`R & D` e 41 de `research and development` por extenso, somadas nos seis
     documentos — o mesmo referente, grafado de duas formas). Sem essa proteção, `R&D`
     colapsaria em dois tokens de uma letra só (`r`, `d`), poluindo o vocabulário com
     ruído não semântico.
   - `U.S.` (abreviação pontuada) → unificado ao mesmo rótulo de `United States`, pela
     mesma razão (evitar os fragmentos `u`, `s`).
   - `science and technology` → rótulo único, por ser expressão fixa recorrente (36
     ocorrências) que a tokenização por adjacência de bigramas não capturaria (o `and`
     no meio quebra a adjacência entre as duas palavras de conteúdo).
3. **Tokenização.** Regex que mantém hífen e apóstrofo internos como parte do token
   (`[A-Za-z][A-Za-z'\-]*[A-Za-z]`), preservando compostos como `decision-making` como
   uma unidade. Todo o texto é convertido para minúsculas.
4. **Remoção do `'s` possessivo** (ex.: `brazil's` → `brazil`, `eu's` → `eu`) — uma
   normalização segura e não controversa (o marcador possessivo não é um conceito
   lexical distinto do substantivo base).

**O que este notebook decide *não* fazer, por transparência:** não há lematização nem
stemming (nenhuma fusão de singular/plural ou de conjugações verbais). Isso evita o
risco de fundir termos com sentidos diferentes sob um mesmo rótulo, mas tem um custo
real, registrado na Seção final de limitações: variantes morfológicas do mesmo lema
(ex.: `intelligentization` vs. `intelligentized`) permanecem como termos distintos ao
longo de toda a análise.

In [ ]:
STOPWORDS_LINE1 = (
    "a about above across after again against all almost along already also although always am among an and "
    "another any anyone anything are around as at be became because become becomes been before being below "
    "between beyond both but by can cannot could did do does doing done down during each either else even ever "
    "every for from further get gets got had has have having he her here hers herself him himself his how i if "
    "in into is it its itself just least less like made make makes many may me might more most much must my "
    "myself neither no nor not now of off often on once only onto or other others our ours ourselves out over "
    "own per rather same shall she should since so some such than that the their theirs them themselves then "
    "there these they this those through thus to too toward towards under until up upon us very via was we were "
    "what whatever when where whereas wherever whether which while who whoever whom whose why will with within "
    "without would yet you your yours yourself yourselves"
)
STOPWORDS = set(STOPWORDS_LINE1.split())
STOPWORDS_CONTRACTIONS = (
    "is's are's isn't aren't wasn't weren't don't doesn't didn't won't wouldn't can't couldn't shouldn't "
    "i'm you're he's she's it's we're they're i've you've we've they've i'll you'll he'll she'll we'll they'll "
    "i'd you'd he'd she'd we'd they'd that's there's who's what's let's"
)
STOPWORDS |= set(STOPWORDS_CONTRACTIONS.split())
STOPWORDS |= {"etc", "eg", "ie", "e", "g", "i", "vs"}

print(f"Lista de stopwords (palavras funcionais do inglês): {len(STOPWORDS)} termos.")

PROTECTED_PHRASES = [
    (r'"?Artificial [Ii]ntelligence\+"?', "AIPLUSTOKEN"),
    (r'AI\+', "AIPLUSTOKEN"),
    (r'R\s*&\s*D', "RANDDTOKEN"),
    (r'[Rr]esearch\s+and\s+[Dd]evelopment', "RANDDTOKEN"),
    (r'\bU\.S\.', "UNITEDSTATESTOKEN"),
    (r'[Ss]cience\s+and\s+[Tt]echnology', "SCITECHTOKEN"),
]
PLACEHOLDER_LABELS = {
    "aiplustoken": "AI+",
    "randdtoken": "R&D",
    "unitedstatestoken": "United States",
    "scitechtoken": "Science & Technology",
}

def normalize_text(text):
    text = text.replace("’", "'").replace("‘", "'").replace("“", '"').replace("”", '"')
    text = text.replace("\xa0", " ")
    for pattern, token in PROTECTED_PHRASES:
        text = re.sub(pattern, f" {token} ", text)
    return text

TOKEN_RE = re.compile(r"[A-Za-z][A-Za-z'\-]*[A-Za-z]|[A-Za-z]")

def strip_possessive(tok):
    if tok.endswith("'s") and len(tok) > 2:
        return tok[:-2]
    return tok

def tokenize_raw(text):
    text = normalize_text(text)
    toks = TOKEN_RE.findall(text)
    return [t.lower() if t.lower() in PLACEHOLDER_LABELS else strip_possessive(t.lower()) for t in toks]

raw_tokens = {k: tokenize_raw(docs[k]["texto"]) for k in doc_keys}
pd.DataFrame({"documento": [docs[k]["label"] for k in doc_keys],
              "tokens brutos (após limpeza de caracteres)": [len(raw_tokens[k]) for k in doc_keys]})

## 2. Detecção de bigramas e expressões compostas

Tratar toda expressão de duas palavras como dois termos isolados fragmenta conceitos
próprios (ex.: separar `open` e `source` perde o conceito "código aberto"). Tratar toda
adjacência como um composto, no outro extremo, infla artificialmente o vocabulário com
pares incidentais sem identidade conceitual própria. Este notebook usa um método híbrido:

1. **Candidatos automáticos por PMI (informação mútua pontual).** Para todo par de
   palavras de conteúdo (nenhuma das duas é stopword) adjacentes em algum dos seis
   documentos, com frequência agregada ≥ 6 no corpus somado, calcula-se
   `PMI(w1,w2) = log2( P(w1,w2) / (P(w1)·P(w2)) )`. PMI alto identifica pares cuja
   coocorrência é muito mais frequente do que o esperado ao acaso — o sinal estatístico
   padrão de uma colocação/expressão fixa.
2. **Limitação conhecida do PMI, tratada à parte.** PMI penaliza sistematicamente
   bigramas que envolvem uma palavra extremamente frequente — e `ai` é, disparadamente,
   a palavra mais frequente de todo o corpus. Isso faz com que compostos claramente
   relevantes (`generative ai`, `ai act`, `frontier ai`, `ai governance`) apareçam com
   PMI comparativamente baixo, não por serem menos importantes, mas por causa da alta
   frequência marginal de `ai`. Por isso, todos os pares candidatos envolvendo `ai`
   (71 pares distintos, frequência ≥ 6) foram inspecionados **separadamente**, por
   frequência bruta, e não descartados apenas por PMI baixo.
3. **Decisão caso a caso.** Cada candidato foi avaliado por leitura de contexto: vira
   termo único apenas quando denota um conceito, mecanismo ou nome próprio com
   identidade analítica distinta da soma das partes (ex.: `AI Act`, `generative AI`,
   `open source`, `member states`, `State Council`). Combinações genéricas de modificador
   + `ai` sem identidade própria (`new ai`, `national ai`, `global ai`, `using ai`,
   `promote ai`) foram **rejeitadas deliberadamente** — fundi-las apagaria a centralidade
   real do termo `ai` isoladamente, que é o termo mais importante de todo o corpus por
   definição (todos os seis documentos são, no fundo, sobre IA).

A tabela abaixo mostra os 40 candidatos de maior PMI (frequência ≥ 6) para transparência
do processo — a lista completa de decisões (aceitos/rejeitados) está na célula
seguinte.

In [ ]:
unigram_counts = Counter()
bigram_counts = Counter()
for toks in raw_tokens.values():
    for t in toks:
        if t not in STOPWORDS:
            unigram_counts[t] += 1
    for i in range(len(toks) - 1):
        w1, w2 = toks[i], toks[i+1]
        if w1 not in STOPWORDS and w2 not in STOPWORDS:
            bigram_counts[(w1, w2)] += 1

N = sum(unigram_counts.values())
total_bigrams = sum(bigram_counts.values())
candidates = []
for (w1, w2), c in bigram_counts.items():
    if c < 6:
        continue
    pmi = math.log2((c/total_bigrams) / ((unigram_counts[w1]/N) * (unigram_counts[w2]/N)))
    candidates.append((pmi, c, w1, w2))
candidates.sort(reverse=True)

print(f"Vocabulário de conteúdo bruto (união dos 6 docs): {len(unigram_counts)} termos distintos")
print(f"Pares adjacentes candidatos (freq >= 6): {len(candidates)}\n")
pd.DataFrame([(f"{pmi:.2f}", c, f"{w1} {w2}") for pmi, c, w1, w2 in candidates[:40]],
             columns=["PMI", "freq.", "bigrama candidato"])

In [ ]:
# Decisão final, caso a caso, sobre quais pares viram um único token analítico.
# Critério (documentado acima): concordância + necessidade de identidade conceitual própria.
BIGRAM_MERGES_LIST = [
    ("united","states","United States"), ("open","source","open source"),
    ("machine","learning","machine learning"), ("natural","language","natural language"),
    ("high-performance","computing","high-performance computing"),
    ("high-performance","processing","high-performance processing"),
    ("big","data","big data"), ("swarm","intelligence","swarm intelligence"),
    ("hybrid","enhanced","hybrid enhanced (intelligence)"),
    ("semiconductor","manufacturing","semiconductor manufacturing"),
    ("value","chain","value chain"), ("intellectual","property","intellectual property"),
    ("export","controls","export controls"), ("early","warning","early warning"),
    ("virtual","reality","virtual reality"), ("terminal","devices","terminal devices"),
    ("smart","terminal","smart terminal"), ("automatic","driving","automatic driving"),
    ("autonomous","unmanned","autonomous unmanned"), ("renewable","energy","renewable energy"),
    ("climate","change","climate change"), ("civil","society","civil society"),
    ("single","market","single market"), ("data","spaces","data spaces"),
    ("digital","transformation","digital transformation"), ("higher","education","higher education"),
    ("regulatory","framework","regulatory framework"), ("developing","countries","developing countries"),
    ("innovation","hubs","innovation hubs"), ("public","consultation","public consultation"),
    ("use","cases","use cases"), ("labor","market","labor market"),
    ("information","integrity","information integrity"), ("energy","matrix","energy matrix"),
    ("member","states","member states"), ("best","practices","best practices"),
    ("computational","power","computational power"), ("state","council","State Council"),
    ("trump","administration","Trump administration"), ("executive","order","executive order"),
    ("horizon","europe","Horizon Europe"), ("competitiveness","compass","Competitiveness Compass"),
    ("joint","undertaking","joint undertaking"),
    ("data","center","data center"), ("data","centers","data center"),
    ("data","centre","data center"), ("data","centres","data center"),
    ("public","sector","public sector"), ("private","sector","private sector"),
    ("national","security","national security"),
    ("generative","ai","generative AI"), ("frontier","ai","frontier AI"),
    ("general-purpose","ai","general-purpose AI"), ("ai","literacy","AI literacy"),
    ("ai","governance","AI governance"), ("ai","act","AI Act"),
    ("ai","factories","AI factories"), ("ai","gigafactories","AI gigafactories"),
    ("ai","continent","AI Continent"), ("apply","ai","Apply AI"),
    ("responsible","ai","responsible AI"), ("trustworthy","ai","trustworthy AI"),
    ("sustainable","ai","sustainable AI"), ("ai","talent","AI talent"),
    ("ai","skills","AI skills"), ("ai","ecosystem","AI ecosystem"),
    ("ai","infrastructure","AI infrastructure"), ("ai","strategy","AI strategy"),
    ("critical","infrastructure","critical infrastructure"), ("supply","chain","supply chain"),
    ("foundation","model","foundation model"), ("foundation","models","foundation model"),
    ("large","language","large language (model)"), ("dual","use","dual-use"),
    ("human","oversight","human oversight"), ("risk","management","risk management"),
    ("personal","data","personal data"), ("decision","making","decision-making"),
    ("data","driven","data-driven"), ("cross","border","cross-border"),
    ("human","computer","human-computer"), ("power","grid","power grid"),
    ("industrial","chain","industrial chain"), ("innovation","ecosystem","innovation ecosystem"),
    ("ai","office","AI Office"), ("ai","board","AI Board"), ("ai","alliance","AI Alliance"),
    ("skills","academy","AI Skills Academy"),
]
BIGRAM_MERGES = {(a, b): c for a, b, c in BIGRAM_MERGES_LIST}
print(f"Bigramas/compostos protegidos como termo único: {len(BIGRAM_MERGES)}")
print("Aplicados de forma idêntica aos seis documentos (mesmo par -> mesmo rótulo em qualquer documento onde ocorra).")

def merge_bigrams(tokens):
    out, i, n = [], 0, len(tokens)
    while i < n:
        if i < n - 1 and (tokens[i], tokens[i+1]) in BIGRAM_MERGES:
            out.append(BIGRAM_MERGES[(tokens[i], tokens[i+1])]); i += 2
        else:
            out.append(tokens[i]); i += 1
    return out

def clean_final(tokens):
    out = []
    for t in tokens:
        if t in PLACEHOLDER_LABELS:
            out.append(PLACEHOLDER_LABELS[t]); continue
        if " " in t or any(c.isupper() for c in t):
            out.append(t); continue  # já é rótulo protegido/composto
        if t in STOPWORDS or len(t) <= 1 or t.isdigit():
            continue
        out.append(t)
    return out

merged_tokens = {k: merge_bigrams(v) for k, v in raw_tokens.items()}
final_tokens = {k: clean_final(v) for k, v in merged_tokens.items()}
counts = {k: Counter(final_tokens[k]) for k in final_tokens}
totals = {k: sum(counts[k].values()) for k in counts}

pd.DataFrame({
    "documento": [docs[k]["label"] for k in doc_keys],
    "tokens de conteúdo (após stopwords/bigramas)": [totals[k] for k in doc_keys],
    "termos distintos": [len(counts[k]) for k in doc_keys],
})

## 3. Exclusão de autoidentificadores nacionais/institucionais

Um documento não é mais "distintivo" tematicamente por citar o nome do próprio país,
bloco ou instituição emissora — isso é um efeito mecânico da autoria, não uma escolha de
conteúdo. Se não removidos, esses termos inflam artificialmente a distintividade
vocabular de cada documento na análise de polos (Seção 5) sem refletir nenhuma
especificidade temática real. A decisão é **caso a caso, por documento**, nunca por uma
lista fixa: o mesmo termo pode ser autoidentificador em um documento e conteúdo temático
legítimo em outro.

**Termos excluídos, com justificativa:**

| Documento | Termos excluídos | Natureza |
|---|---|---|
| `pbia` | brazil, brazilian, pbia, mcti, nib, sus, brl | país, nome do próprio plano, instituições (MCTI/CGEE→mcti, NIB), sistema de saúde (SUS) e moeda (BRL) nacionais |
| `ai_plus` | AI+, china, chinese, State Council | nome da própria iniciativa, país, órgão emissor |
| `new_generation` | china, chinese, State Council, rmb | país, órgão emissor, moeda nacional |
| `americas` | United States, american, americans, doc, dod, caisi, nist, doe, nsf, ostp, omb, ic, dol | país e siglas de agências federais dos EUA (Department of Commerce, Defense, CAISI, NIST, Energy, NSF, OSTP, OMB, Intelligence Community, Labor) |
| `apply_ai` | eu, european, europe, commission, Apply AI | bloco, instituição emissora (Comissão Europeia), nome da própria estratégia |
| `ai_continent` | eu, european, europe, commission, AI Continent | idem, nome do próprio plano |

**Casos de fronteira, avaliados e *mantidos* (não excluídos), com justificativa
individual — mesma lógica de julgamento caso a caso, aplicada na direção inversa:**
- `china`/`chinese` em `americas` (2 ocorrências) e em `ai_continent` (1 ocorrência) —
  referência a um terceiro país/concorrente estratégico, não autoidentificação; carrega
  conteúdo temático real (enquadramento de rivalidade geopolítica).
- `Trump`/`Trump administration` em `americas` — nome de uma pessoa/governo específico,
  não literalmente um nome de país ou instituição; mantido por conservadorismo
  metodológico (é um dado comparativo real: qual governo assina o documento).
- `Apply AI` em `ai_continent` (referência à estratégia irmã) e `AI Continent` em
  `apply_ai` (idem) — mantidos como conteúdo, não excluídos: cada termo só é
  autoidentificador **do documento que leva esse nome**, não do documento vizinho que
  meramente o cita. Ressalva: como as duas estratégias são publicadas pela mesma
  Comissão Europeia, uma leitura mais rígida poderia tratar essas referências cruzadas
  como quase-autoidentificação também — registrado como limitação na seção final.
- Nomes de programas/instrumentos específicos da UE mencionados como conteúdo
  (`Horizon Europe`, `Competitiveness Compass`, `EuroHPC`, `InvestAI`) e de
  Estados-membros citados como exemplo (`germany`, `spain`, `estonia`) — mantidos:
  descrevem o instrumental de política concreto do bloco, não são autonomeação da UE
  como um todo.

In [ ]:
SELF_ID_EXCLUDE = {
    "pbia": ["brazil","brazilian","pbia","mcti","nib","sus","brl"],
    "ai_plus": ["AI+","china","chinese","State Council"],
    "new_generation": ["china","chinese","State Council","rmb"],
    "americas": ["United States","american","americans","doc","dod","caisi","nist","doe","nsf","ostp","omb","ic","dol"],
    "apply_ai": ["eu","european","europe","commission","Apply AI"],
    "ai_continent": ["eu","european","europe","commission","AI Continent"],
}

rows = []
final_counts, final_totals = {}, {}
for k in doc_keys:
    excl = sum(counts[k].get(t, 0) for t in SELF_ID_EXCLUDE[k])
    rows.append({"documento": docs[k]["label"], "ocorrências excluídas": excl,
                 "% do vocabulário bruto de conteúdo": round(excl/totals[k]*100, 2)})
    final_counts[k] = Counter({t: n for t, n in counts[k].items() if t not in SELF_ID_EXCLUDE[k]})
    final_totals[k] = sum(final_counts[k].values())

exclusion_df = pd.DataFrame(rows)
exclusion_df

A exclusão remove entre **0,4% e 5,0%** do vocabulário bruto de cada documento — uma
fração pequena, mas concentrada nos termos mais mecanicamente óbvios de autoidentificação
(ex.: "United States" sozinho responde por 50 das 256 ocorrências excluídas do
`americas`). A partir daqui, **todas** as porcentagens e classificações do notebook usam
`final_counts`/`final_totals` (vocabulário de conteúdo pós-exclusão) como base — nunca o
vocabulário bruto.

## 4. Vocabulário Consensual — definição e sensibilidade do limiar

**Definição operacional.** Um termo é *consensual* quando aparece em pelo menos 4 dos 6
documentos (`df ≥ 4`, onde `df` é o número de documentos, entre os 6, que contêm o
termo) — a leitura literal de "o vocabulário que a maioria dos textos converge": maioria
de 6 documentos é, no mínimo, 4. O restante (`df ≤ 3`) é o **vocabulário distintivo**,
candidato aos quatro polos A/B/C/D (Seção 5).

Antes de fixar esse limiar, a tabela abaixo mostra a sensibilidade da escolha — quantos
termos e quanta massa de vocabulário (frequência relativa média, por documento) cada
corte alternativo capturaria:

In [ ]:
rel_freq = {k: {t: n/final_totals[k] for t, n in final_counts[k].items()} for k in final_counts}
vocab_union = set().union(*[set(final_counts[k].keys()) for k in final_counts])
df_count = Counter()
total_count_corpus = Counter()
for t in vocab_union:
    for k in final_counts:
        if t in final_counts[k]:
            df_count[t] += 1
            total_count_corpus[t] += final_counts[k][t]

rows = []
for thr in [3, 4, 5, 6]:
    terms_thr = [t for t in vocab_union if df_count[t] >= thr]
    mass_by_doc = [sum(rel_freq[k].get(t, 0) for t in terms_thr) for k in doc_keys]
    rows.append({"limiar (df >=)": thr, "nº de termos": len(terms_thr),
                 "cobertura média da massa/doc": f"{np.mean(mass_by_doc)*100:.1f}%",
                 "mín.": f"{min(mass_by_doc)*100:.1f}%", "máx.": f"{max(mass_by_doc)*100:.1f}%"})
sensitivity_df = pd.DataFrame(rows)
print(f"Vocabulário-união dos 6 documentos (pós-exclusão): {len(vocab_union)} termos distintos.\n")
sensitivity_df

In [ ]:
CONSENSUAL_THR = 4
consensual_terms = set(t for t in vocab_union if df_count[t] >= CONSENSUAL_THR)
distinctive_terms = set(t for t in vocab_union if df_count[t] <= 3)
print(f"Vocabulário CONSENSUAL (df >= {CONSENSUAL_THR}): {len(consensual_terms)} termos "
      f"({sensitivity_df.loc[sensitivity_df['limiar (df >=)']==CONSENSUAL_THR,'cobertura média da massa/doc'].iloc[0]} da massa média de cada documento)")
print(f"Vocabulário DISTINTIVO (df <= 3): {len(distinctive_terms)} termos, candidato aos polos A/B/C/D")

O limiar `df ≥ 4` foi escolhido por corresponder à leitura literal de "maioria dos
textos" e por marcar a maior queda relativa de cobertura na tabela de sensibilidade
(de 76,8% em `df≥3` para 63,9% em `df≥4`, depois uma queda mais suave até `df≥6`) — ou
seja, é o ponto em que a fração "compartilhada pela maioria" já capturou a maior parte do
vocabulário genuinamente comum, sem ainda exigir unanimidade estrita (`df=6`), que
excluiria termos presentes em 4 ou 5 dos 6 documentos apenas por não estarem no sexto.
Essa é uma escolha declarada, não a única defensável — a tabela acima permite ao leitor
avaliar como o resultado mudaria com `df≥3` ou `df≥5`.

## 5. Vocabulário distintivo → polos A/B/C/D — metodologia

Esta é a etapa metodologicamente mais sensível do notebook. O objetivo é reduzir os
termos distintivos (4.354 termos, presentes em no máximo 3 dos 6 documentos) a **duas
dimensões** — os eixos da bússola —, de modo que tanto os documentos quanto os próprios
termos possam ser posicionados no mesmo espaço.

**Por que não Análise de Correspondência (CA) — o método clássico para este tipo de
tabela termo×documento.** CA pondera documentos pela sua massa marginal (tamanho),
dando a documentos maiores mais peso na definição dos eixos — o que violaria diretamente
a exigência de comparação relativa com peso igual entre documentos de tamanhos
diferentes (ver Introdução). Este notebook usa, em vez disso, **PCA sobre o perfil de
frequência relativa, com cada um dos 6 documentos como uma observação de peso igual**
(1/6) — uma adaptação deliberada que troca a ortodoxia estatística da CA pelo requisito
explícito de peso igual por documento.

**Passo a passo:**

1. **Filtro de frequência mínima.** Dos 4.354 termos distintivos, apenas os
   **1.403 com contagem total ≥ 3 no corpus somado** entram no cálculo dos eixos
   (`confident_terms`) — termos que ocorrem só 1 ou 2 vezes em todo o corpus não têm
   sinal estatístico suficiente para ajudar a definir uma direção estável, e entrariam
   como ruído. Os 2.951 termos restantes (a cauda de frequência 1–2) são classificados
   depois, por uma regra separada (passo 5).
2. **Transformação.** Cada frequência relativa passa por raiz quadrada
   (`sqrt`, transformação estabilizadora de variância padrão para dados de contagem) —
   sem isso, um único termo de frequência muito alta dominaria sozinho a variância e os
   eixos resultantes; com a transformação, tanto termos moderadamente frequentes quanto
   termos mais raros (mas ainda com `contagem >= 3`) contribuem de forma proporcional,
   sem que um punhado de termos de altíssima frequência apague o resto do sinal.
3. **Centralização (sem padronização para variância unitária).** As colunas são
   centralizadas (média subtraída), mas *não* escalonadas para variância 1 — decisão
   deliberada: padronizar daria a um termo de ocorrência única e isolada o mesmo peso
   estatístico que a um termo consistentemente relevante, exatamente o problema que a
   etapa 1 já mitiga por outro caminho.
4. **Decomposição (SVD/PCA).** Sobre a matriz 6×1.403 assim construída, `numpy.linalg.svd`
   fornece, ao mesmo tempo, a posição de cada um dos 6 documentos (`PC1`, `PC2`) e a
   carga (*loading*) de cada um dos 1.403 termos nesses mesmos dois eixos — documentos e
   termos compartilham o mesmo espaço, o que é o que permite plotar os dois na mesma
   bússola (Seções 7 e 8).
5. **Termos de cauda (contagem total 1–2, 2.951 termos).** Não entram no cálculo dos
   eixos, mas ainda precisam de uma classificação para a tabela de composição (Seção 6),
   sob pena de deixar de fora uma fração do vocabulário. Regra de atribuição, mais fraca
   e declarada como tal: um termo de cauda é atribuído ao polo predominante do
   documento onde ocorre com maior frequência relativa entre os poucos documentos em que
   aparece — **não** ao seu próprio *loading* (que não existe, por não ter entrado no
   PCA). Como mostra a Seção 6, essa cauda de baixíssima frequência responde por uma
   fração pequena da massa total de cada documento.
6. **Convenção de orientação dos polos.** O sinal de PC1 e de PC2 é matematicamente
   arbitrário (PCA não define "para que lado" cada eixo aponta). A convenção usada aqui,
   só para fins de rotulagem visual — sem qualquer significado substantivo — é:
   **D = PC1 positivo, B = PC1 negativo, A = PC2 positivo, C = PC2 negativo.**
7. **Classificação termo→polo ("regra do eixo dominante").** Cada termo (entre os 1.403
   com carga calculada) é atribuído ao polo do eixo em que sua carga tem **maior valor
   absoluto** — equivalente a dividir o plano em quatro quadrantes de 90° centrados em
   cada eixo cardeal. Um termo com carga muito maior em PC1 do que em PC2 vai para B ou D
   (conforme o sinal); o inverso vai para A ou C.

In [ ]:
MIN_TOTAL_COUNT = 3
confident_terms = sorted([t for t in distinctive_terms if total_count_corpus[t] >= MIN_TOTAL_COUNT])
tail_terms = sorted([t for t in distinctive_terms if total_count_corpus[t] < MIN_TOTAL_COUNT])
print(f"Termos distintivos usados para definir os eixos via PCA (contagem total >= {MIN_TOTAL_COUNT}): {len(confident_terms)}")
print(f"Termos de cauda, classificados por regra de fallback (contagem total 1-2): {len(tail_terms)}")

X = np.array([[rel_freq[k].get(t, 0.0) for t in confident_terms] for k in doc_keys])
Xs = np.sqrt(X)
Xc = Xs - Xs.mean(axis=0)

U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
doc_scores = U * S
var_explained = (S**2) / np.sum(S**2)
PC1, PC2 = doc_scores[:, 0], doc_scores[:, 1]
term_loadings = Vt.T[:, :2] * S[:2]

print(f"\nVariância explicada por componente: {np.round(var_explained*100,1)}")
print(f"PC1 + PC2 (os dois eixos da bússola) capturam {round((var_explained[0]+var_explained[1])*100,1)}% da variância total de distância entre os 6 documentos.")

pos_df = pd.DataFrame({
    "documento": [docs[k]["label"] for k in doc_keys],
    "país/bloco": [docs[k]["pais_ou_bloco"] for k in doc_keys],
    "PC1": np.round(PC1, 4), "PC2": np.round(PC2, 4),
})
pos_df

**Leitura honesta do ajuste.** PC1+PC2 capturam **53,7%** da variância total de
distância entre os 6 documentos — os outros ~46% estão em PC3, PC4 e PC5 (que existem,
já que 6 documentos permitem até 5 componentes não triviais), mas não são representados
na bússola bidimensional. Isso é comparável em ordem de grandeza ao ajuste reportado no
mapa MDS de `analise_geral.ipynb` (58,3%, com stress-1 "pobre" por outro método) — ambos
os notebooks, por caminhos diferentes, chegam à mesma conclusão qualitativa: reduzir 6
documentos a um mapa 2D é informativo para agrupamento amplo, mas descarta uma fração
substancial da estrutura real. Tratar a posição de qualquer documento na bússola como
uma medida exata, e não como uma projeção aproximada, seria uma imprecisão metodológica.

In [ ]:
def classify_dominant_axis(pc1, pc2):
    if abs(pc1) >= abs(pc2):
        return "D" if pc1 > 0 else "B"
    return "A" if pc2 > 0 else "C"

idx_of = {t: i for i, t in enumerate(confident_terms)}
term_pole, term_confidence = {}, {}
for t in confident_terms:
    p1, p2 = term_loadings[idx_of[t], 0], term_loadings[idx_of[t], 1]
    term_pole[t] = classify_dominant_axis(p1, p2)
    term_confidence[t] = math.hypot(p1, p2)

doc_home_pole = {k: classify_dominant_axis(PC1[i], PC2[i]) for i, k in enumerate(doc_keys)}

for t in tail_terms:
    best_doc = max(doc_keys, key=lambda k: rel_freq[k].get(t, 0.0))
    term_pole[t] = doc_home_pole[best_doc]
    term_confidence[t] = None

pd.DataFrame({
    "documento": [docs[k]["label"] for k in doc_keys],
    "polo predominante (quadrante)": [doc_home_pole[k] for k in doc_keys],
})

In [ ]:
pole_dist = Counter(term_pole.values())
print("Distribuição dos termos distintivos entre os 4 polos:")
for p in ["A","B","C","D"]:
    print(f"  Vocabulário {p}: {pole_dist[p]:5d} termos")
print("\nNota: a distribuição NÃO é, e não deveria ser esperada como, equilibrada entre os quatro polos —")
print("é um resultado empírico, não um artefato do método (ver interpretação na Seção 7).")

## 6. Composição de cada documento nas 5 categorias

Para cada documento, soma-se a frequência relativa de todos os seus termos que caem em
cada categoria (Consensual, A, B, C, D). Como cada termo do vocabulário de conteúdo
pós-exclusão pertence a exatamente uma das cinco categorias (partição completa, sem
sobreposição), as cinco frações somam exatamente 100% para cada documento — a checagem
de consistência mais direta de que a classificação está completa e não tem lacunas nem
dupla contagem.

In [ ]:
categories = ["Consensual", "A", "B", "C", "D"]
mass_table = {k: {c: 0.0 for c in categories} for k in doc_keys}
for k in doc_keys:
    for t, f in rel_freq[k].items():
        cat = "Consensual" if t in consensual_terms else term_pole[t]
        mass_table[k][cat] += f

comp_df = pd.DataFrame({
    "Documento": [f"{docs[k]['label']} ({docs[k]['pais_ou_bloco']})" for k in doc_keys],
    **{f"{c} (%)": [round(mass_table[k][c]*100, 2) for k in doc_keys] for c in categories},
})
comp_df["Soma (%)"] = comp_df[[f"{c} (%)" for c in categories]].sum(axis=1).round(2)
comp_df

**Checagem de consistência:** a coluna "Soma (%)" é 100,00% em todas as seis linhas —
confirma que a partição Consensual/A/B/C/D é completa (todo termo do vocabulário de
conteúdo pós-exclusão pertence a exatamente uma categoria) e que nenhuma massa de
vocabulário foi perdida ou contada em duplicidade entre as etapas anteriores.

## 7. A bússola de vocabulário — posição de cada documento

**Leitura do gráfico.** Cada bolha é um documento, posicionado por (PC1, PC2) — a
projeção de duas dimensões do seu vocabulário distintivo (Seção 5). O **tamanho** da
bolha é a fração do vocabulário do documento que é distintivo (100% − % Consensual da
Seção 6): bolhas maiores são documentos vocabularmente mais idiossincráticos; bolhas
menores usam, proporcionalmente, mais do vocabulário-padrão compartilhado. A **cor**
identifica o país/bloco (Brasil, China, Estados Unidos, Europa).

In [ ]:
bloc_color = {"Brasil": COL_BLUE, "China": COL_ORANGE, "Estados Unidos": COL_AQUA, "Europa (UE)": COL_YELLOW}
doc_label = {k: docs[k]["label"] for k in doc_keys}

LABEL_OFFSET = {
    "pbia":            (0.028,  0.000, "left",   "center"),
    "ai_plus":         (0.000,  0.040, "center", "bottom"),
    "new_generation":  (0.000, -0.040, "center", "top"),
    "americas":        (0.000, -0.035, "center", "top"),
    "apply_ai":        (0.000, -0.040, "center", "top"),
    "ai_continent":    (0.000,  0.040, "center", "bottom"),
}

def pole_box(ax, x, y, text, ha, va):
    ax.annotate(text, xy=(x, y), xycoords="data", ha=ha, va=va, fontsize=11, fontweight="bold",
                color="white", bbox=dict(boxstyle="round,pad=0.45", fc=POLE_BG, ec="none"), zorder=6)

fig, ax = plt.subplots(figsize=(9.5, 7.5), dpi=130)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
lim = max(np.abs(PC1).max(), np.abs(PC2).max()) * 1.55
ax.axhline(0, color=AXIS, lw=1.4, zorder=1); ax.axvline(0, color=AXIS, lw=1.4, zorder=1)
pole_box(ax, 0, lim*0.97, "Vocabulário A", "center", "top")
pole_box(ax, 0, -lim*0.97, "Vocabulário C", "center", "bottom")
pole_box(ax, -lim*0.97, 0, "Vocabulário B", "left", "center")
pole_box(ax, lim*0.97, 0, "Vocabulário D", "right", "center")

sizes = np.array([(1 - mass_table[k]["Consensual"]) for k in doc_keys])
size_scale = 3400
for i, k in enumerate(doc_keys):
    bloc = docs[k]["pais_ou_bloco"]; color = bloc_color[bloc]
    ax.scatter(PC1[i], PC2[i], s=sizes[i]*size_scale, color=color, alpha=0.55, edgecolor=color, linewidth=1.8, zorder=3)
    ax.scatter(PC1[i], PC2[i], s=20, color=color, zorder=4)
    dx, dy, ha, va = LABEL_OFFSET[k]
    ax.annotate(f"{doc_label[k]}\n({bloc}) · {sizes[i]*100:.0f}% distintivo",
                xy=(PC1[i], PC2[i]), xytext=(PC1[i]+dx, PC2[i]+dy),
                fontsize=8.2, color=INK_PRIMARY, ha=ha, va=va, linespacing=1.35, zorder=5)

ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)
ax.set_title("Bússola de vocabulário — posição de cada documento (PC1 × PC2)\ntamanho da bolha = % do vocabulário do documento que é distintivo (não consensual)",
             fontsize=10.5, color=INK_SECONDARY, pad=14)
plt.tight_layout()
plt.show()

**Interpretação, estritamente dentro do que o gráfico sustenta:**

- **Os dois documentos chineses** (`"AI+" Initiative` e `New Generation AI Development
  Plan`) ocupam o mesmo quadrante (D), próximos entre si — coerência interna do bloco
  chinês neste espaço vocabular, mesmo com 8 anos de distância entre os dois planos.
- **Os dois documentos europeus** (`Apply AI Strategy` e `AI Continent Action Plan`)
  ocupam o quadrante oposto no eixo horizontal (B), também próximos entre si — o eixo
  PC1 (B↔D) separa nitidamente o polo chinês do polo europeu.
- **America's AI Action Plan** é o ponto mais extremo do mapa inteiro, isolado no polo C
  — um achado desta metodologia especificamente, não reproduzido de `analise_geral.ipynb`
  (onde o PBIA era o outlier mais citado). A composição por categoria (Seção 6) mostra
  que **36,9%** do vocabulário deste documento é do tipo C, a maior concentração
  individual de qualquer documento em qualquer polo.
- **O PBIA fica próximo do centro** no eixo horizontal (quase não pende nem para B nem
  para D) e moderadamente no polo C — mais perto, nesse sentido específico, do
  `America's AI Action Plan` do que dos outros quatro documentos, mas de forma bem menos
  extrema.
- **Tamanho das bolhas:** varia relativamente pouco (30%–43% de vocabulário distintivo),
  com o `America's AI Action Plan` no extremo mais idiossincrático (43%) e a `"AI+"
  Initiative` no mais próximo do vocabulário-padrão (30%, mas ainda assim quase um terço
  do seu vocabulário).

**Limitações específicas deste gráfico:**
- A posição é uma projeção que preserva 53,7% da variância de distância — tratá-la como
  medida exata de proximidade entre dois documentos específicos (em vez de uma leitura
  aproximada de agrupamento) seria uma imprecisão sobre o que o método garante.
- O polo A é visivelmente menos "ocupado" que B, C e D neste mapa (nenhum documento tem
  A como polo predominante) — não há garantia estatística de que os quatro polos sejam
  simetricamente relevantes; ver Seção 9 para a distribuição de termos por polo, que
  confirma isso quantitativamente.
- Com apenas 6 documentos, qualquer um deles tem influência substancial sobre a forma
  final do mapa — o resultado não deve ser lido como um padrão estatisticamente
  generalizável para "documentos chineses" ou "documentos europeus" em geral, apenas
  para estes seis textos específicos.

## 8. A bússola de vocabulário — termos representativos de cada polo

O mesmo espaço (PC1, PC2), agora com os **termos** — não os documentos — plotados em
suas posições (*loadings*). Mostra literalmente quais palavras "puxam" cada polo,
respondendo de forma direta à pergunta "quais vocabulários se encaixam em cada
categoria". Critério de seleção: até 8 termos por polo, os de **maior carga (magnitude)**
— ou seja, os que mais fortemente definem aquele eixo, não necessariamente os de maior
frequência bruta (esses últimos estão nas tabelas da Seção 9).

In [ ]:
pole_color = {"A": COL_AQUA, "B": COL_BLUE, "C": COL_RED, "D": COL_ORANGE}
agg_mass = {t: sum(rel_freq[k].get(t, 0.0) for k in doc_keys) for t in vocab_union}

TOP_N_TERMS = 8
fig, ax = plt.subplots(figsize=(12, 9.5), dpi=130)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)

mag_sorted = sorted(confident_terms, key=lambda t: -term_confidence[t])
outlier_terms = []
if term_confidence[mag_sorted[0]] > 1.4 * term_confidence[mag_sorted[1]]:
    outlier_terms = [mag_sorted[0]]
plot_pool = [t for t in confident_terms if t not in outlier_terms]
tlim = max(abs(term_loadings[idx_of[t], 0]) for t in plot_pool)
tlim = max(tlim, max(abs(term_loadings[idx_of[t], 1]) for t in plot_pool)) * 1.5

ax.axhline(0, color=AXIS, lw=1.2, zorder=1); ax.axvline(0, color=AXIS, lw=1.2, zorder=1)
pole_box(ax, 0, tlim*0.97, "Vocabulário A", "center", "top")
pole_box(ax, 0, -tlim*0.97, "Vocabulário C", "center", "bottom")
pole_box(ax, -tlim*0.97, 0, "Vocabulário B", "left", "center")
pole_box(ax, tlim*0.97, 0, "Vocabulário D", "right", "center")

def stagger_labels(points):
    out = []
    for i, (t, x, y) in enumerate(points):
        extra = 1.0 + (i % 3) * 0.11
        out.append((t, x, y, extra))
    return out

for pole in ["A", "B", "C", "D"]:
    terms_p = [t for t in plot_pool if term_pole[t] == pole]
    terms_p.sort(key=lambda t: -term_confidence[t])
    top_terms = terms_p[:TOP_N_TERMS]
    pts = sorted([(t, term_loadings[idx_of[t], 0], term_loadings[idx_of[t], 1]) for t in top_terms], key=lambda p: p[1])
    pts = stagger_labels(pts)
    ax.scatter([p[1] for p in pts], [p[2] for p in pts], s=28, color=pole_color[pole], zorder=4, alpha=0.9)
    for t, x, y, extra in pts:
        ax.annotate(t, xy=(x, y), xytext=(x*extra, y*extra + (0.006 if y >= 0 else -0.006)),
                    fontsize=8.3, color=INK_PRIMARY, ha="center", va="bottom" if y >= 0 else "top", zorder=5)

ax.set_xlim(-tlim, tlim); ax.set_ylim(-tlim, tlim); ax.set_xticks([]); ax.set_yticks([])
for s in ax.spines.values(): s.set_visible(False)
subtitle = f"Bússola de vocabulário — até {TOP_N_TERMS} termos de maior carga (loading) em cada polo"
if outlier_terms:
    ratio = term_confidence[outlier_terms[0]] / term_confidence[mag_sorted[1]]
    subtitle += f"\n(\"{outlier_terms[0]}\" fica fora de escala — carga {ratio:.1f}x maior que o 2º termo mais forte; ver tabelas da Seção 9)"
ax.set_title(subtitle, fontsize=10, color=INK_SECONDARY, pad=14)
plt.tight_layout()
plt.show()

**Leitura, termo a termo, dos quatro polos:**

- **Vocabulário D** (polo dos dois documentos chineses): `intelligent` (fora de escala —
  ver nota no gráfico e Seção 9), `robots`, `devices`, `intelligentization`,
  `intelligentized`, `smart terminal`, `formation`, `upgrading`. Um agrupamento
  claramente ancorado em vocabulário de engenharia/hardware e no estilo de tradução
  institucional chinesa (`intelligentization`, `vigorously` — ver Seção 9 — não são
  formas comuns do inglês de política pública ocidental).
- **Vocabulário B** (polo dos dois documentos europeus): `AI Act`, `member states`,
  `AI factories`, `public sector`, `smes`, `context`. Vocabulário regulatório e
  institucional específico da arquitetura de governança da UE.
- **Vocabulário C** (polo do `America's AI Action Plan`, com contribuição do PBIA):
  `federal`, `led`, `recommended`, `america`, `department`, `programs`. Vocabulário de
  estrutura de governo federal e de linguagem de recomendação/ação — ver a ressalva sobre
  efeito de formato na Seção 10.
- **Vocabulário A** (o polo menos povoado — 159 termos, contra 1.024–1.848 nos outros
  três): `agents`, `reasoning`, `push`, `pool`, `factory`, `programmes`. Não forma um
  agrupamento temático tão nitidamente coerente quanto B, C ou D — é, com os dados
  disponíveis, o polo mais fraco e mais heterogêneo dos quatro (ver Seção 9 para a
  discussão quantitativa dessa assimetria).

## 9. Composição relativa — gráfico de barras 100% empilhadas

A mesma tabela da Seção 6, em forma visual. Cada barra soma exatamente 100% — é o
gráfico que responde de modo mais direto e sem projeção/perda de variância à pergunta
"o quanto de cada documento pertence a cada vocabulário", já que (diferentemente da
bússola das Seções 7-8) não depende de reduzir nada a duas dimensões.

In [ ]:
order = ["ai_continent", "apply_ai", "americas", "new_generation", "ai_plus", "pbia"]
stack_categories = ["Consensual", "B", "D", "A", "C"]
stack_colors = {"Consensual": INK_MUTED, "A": COL_AQUA, "B": COL_BLUE, "C": COL_RED, "D": COL_ORANGE}
labels_y = [f"{doc_label[k]}\n({docs[k]['pais_ou_bloco']})" for k in order]

fig, ax = plt.subplots(figsize=(9.5, 5), dpi=130)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)
left = np.zeros(len(order))
for cat in stack_categories:
    vals = np.array([mass_table[k][cat]*100 for k in order])
    ax.barh(labels_y, vals, left=left, color=stack_colors[cat],
            label=("Consensual" if cat == "Consensual" else f"Vocabulário {cat}"),
            height=0.62, edgecolor=SURFACE, linewidth=2)
    for i, (v, l) in enumerate(zip(vals, left)):
        if v > 3.5:
            ax.text(l+v/2, i, f"{v:.1f}%", ha="center", va="center", fontsize=7.6,
                    color="white" if cat != "Consensual" else INK_PRIMARY, fontweight="bold")
    left += vals

ax.set_xlim(0, 100)
ax.set_xlabel("% do vocabulário de conteúdo do documento (base = pós-exclusão de autoidentificadores)", fontsize=8.7, color=INK_SECONDARY)
ax.tick_params(axis="y", labelsize=8.3, colors=INK_PRIMARY)
ax.tick_params(axis="x", labelsize=7.8, colors=INK_MUTED)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
for s in ["left", "bottom"]: ax.spines[s].set_color(AXIS)
ax.set_title("Composição relativa de cada documento — Consensual vs. A/B/C/D", fontsize=11, color=INK_PRIMARY, pad=12)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=5, frameon=False, fontsize=8.3)
plt.tight_layout()
plt.show()

**Leitura do gráfico:** em todos os seis documentos, o vocabulário **Consensual** é a
fatia majoritária — entre 57,5% (`America's AI Action Plan`) e 69,7% (`"AI+"
Initiative`). Isso é esperado e não é um artefato: os seis documentos tratam do mesmo
campo de política pública, então a maior parte do vocabulário de qualquer um deles
deveria mesmo ser compartilhada. O que diferencia os documentos é a composição do
**restante** — e aí o padrão por bloco é nítido: os dois documentos chineses concentram
sua parcela distintiva quase inteiramente em D (27,0% e 24,6%); os dois europeus,
quase inteiramente em B (28,8% e 31,5%); o `America's AI Action Plan` concentra 36,9% em
C sozinho — a maior concentração de qualquer documento em qualquer polo único do corpus;
o PBIA é o mais distribuído entre polos (C 23,1%, B 8,8%, D 5,3%, A 1,6%), sem uma
concentração dominante única, o que é coerente com sua posição próxima ao centro da
bússola na Seção 7.

## 10. Quais são os vocabulários — tabelas completas por categoria

Os termos de maior massa agregada (soma das frequências relativas dos 6 documentos, cada
um com peso igual) em cada uma das cinco categorias. Cada linha mostra também em quantos
e em quais documentos o termo ocorre, para que a tabela seja auditável, não apenas uma
lista de palavras soltas.

In [ ]:
term_category = {t: ("Consensual" if t in consensual_terms else term_pole[t]) for t in vocab_union}

def top_table(cat, n=25):
    terms_in_cat = sorted([t for t in vocab_union if term_category[t] == cat], key=lambda t: -agg_mass[t])[:n]
    rows = []
    for t in terms_in_cat:
        present_in = [doc_label[k] for k in doc_keys if t in final_counts[k]]
        rows.append({"termo": t, "massa agregada (%)": round(agg_mass[t]*100, 3),
                     "em quantos docs (de 6)": len(present_in), "quais documentos": ", ".join(present_in)})
    return pd.DataFrame(rows)

print(f"Vocabulário CONSENSUAL — {len(consensual_terms)} termos no total. Top 25:")
top_table("Consensual")

In [ ]:
print(f"Vocabulário A — {pole_dist['A']} termos no total (o polo menos povoado). Top 25:")
top_table("A")

In [ ]:
print(f"Vocabulário B — {pole_dist['B']} termos no total. Top 25:")
top_table("B")

In [ ]:
print(f"Vocabulário C — {pole_dist['C']} termos no total. Top 25:")
top_table("C")

In [ ]:
print(f"Vocabulário D — {pole_dist['D']} termos no total. Top 25:")
top_table("D")

**Sobre o termo `intelligent` (Vocabulário D).** É, disparadamente, o termo distintivo
de maior carga de todo o corpus (3,25% da massa agregada — mais que o dobro do segundo
colocado da mesma categoria, `compute`, com 0,68%), presente nos dois documentos
chineses e no PBIA, ausente nos três documentos ocidentais restantes. Somado a
`intelligentization`/`intelligentized` (também no Vocabulário D, sem lematização — ver
Seção 1), esse é o sinal lexical mais forte de todo o notebook, e reflete tanto uma
escolha real de enquadramento tecnológico quanto, possivelmente, convenções de tradução
do chinês institucional para o inglês — as duas explicações não são distinguíveis apenas
a partir deste gráfico (ver Seção 11).

## 11. Limitações, imprecisões conhecidas e sugestões para trabalho futuro

### Limitações estruturais dos dados

1. **Índia ausente.** Não há `.json` extraído para o plano nacional indiano — a análise
   cobre os 6 documentos disponíveis, não os 5 países/blocos "completos" do projeto.
   Qualquer leitura deste notebook como "os 5 blocos do projeto" seria uma generalização
   indevida; é "os 6 documentos atualmente extraídos".
2. **Tradução como limitação estrutural herdada.** Os dois documentos chineses e, em
   menor medida, o PBIA chegam ao corpus como texto em inglês produzido por tradução
   institucional (CSET, New America/DigiChina) ou por publicação oficial já em inglês —
   nenhuma etapa deste notebook elimina esse efeito, apenas o registra. O forte peso de
   `intelligent`/`intelligentization`/`vigorously` no Vocabulário D (Seção 10) pode
   refletir, em parte, convenção de tradução, não apenas escolha de conteúdo — os dados
   aqui reunidos não permitem separar as duas explicações.
3. **Efeito de gênero/formato textual, não apenas de conteúdo.** O `America's AI Action
   Plan` é organizado em itens de "Recommended Policy Action" numerados e repetidos —
   isso pode inflar artificialmente termos como `recommended`, `federal`, `led` no
   Vocabulário C (Seção 10) por estrutura documental, não só por ênfase temática. O mesmo
   efeito, já registrado em `pbia_vocab_registro.md` para o formato do PBIA (planos
   extensos com itens numerados), pode estar presente aqui também, de forma menos
   marcada. Este notebook identifica o risco, mas não o separa estatisticamente do sinal
   temático.

### Limitações do pipeline de pré-processamento (Seções 1-3)

4. **Sem lematização.** Variantes morfológicas do mesmo lema (`intelligentization` vs.
   `intelligentized`; `theory` vs. `theories`) permanecem como termos distintos em todas
   as tabelas e gráficos — a massa real de um mesmo conceito está subestimada e dividida
   entre entradas separadas sempre que isso ocorre.
5. **Detecção de bigramas não é uma varredura exaustiva.** A lista de 87 bigramas
   protegidos (Seção 2) cobriu os candidatos de frequência ≥ 6 mais os 71 pares
   envolvendo `ai`; candidatos de frequência 3-5 não foram revisados individualmente —
   uma fração desconhecida, mas provavelmente pequena, do vocabulário distintivo de
   baixa frequência pode conter compostos não identificados como tal.
6. **Exclusão de autoidentificadores (Seção 3) não é exaustiva.** Cobriu os casos de
   frequência alta/média mais uma busca direcionada por nomes de país/bloco/instituição
   — não uma varredura termo a termo de toda a cauda longa (750-2.400 termos distintos
   por documento). O caso de fronteira sobre `Apply AI`/`AI Continent` como possível
   quase-autoidentificação cruzada (mesma Comissão Europeia) está registrado na Seção 3,
   não resolvido.

### Limitações do método estatístico (Seções 4-5)

7. **O limiar `df≥4` para "consensual" é uma escolha declarada, não a única válida** — a
   tabela de sensibilidade da Seção 4 permite ao leitor avaliar `df≥3` ou `df≥5` como
   alternativas; os polos A/B/C/D mudariam de composição (mas não devem mudar
   qualitativamente de orientação) sob um limiar diferente, algo que este notebook não
   testou exaustivamente.
8. **PC1+PC2 capturam 53,7% da variância de distância** (Seção 5) — quase metade da
   estrutura real entre os 6 documentos não está representada nas Seções 7 e 8. A
   posição relativa de dois documentos próximos na bússola pode não refletir com
   fidelidade sua real proximidade nas dimensões não capturadas (PC3-PC5).
9. **N=6 documentos é uma amostra pequena** para qualquer PCA — cada documento
   individual tem influência substancial sobre a forma final dos eixos; o resultado é
   específico a estes seis textos, não generalizável estatisticamente a "documentos
   chineses" ou "documentos europeus" de política de IA em geral.
10. **Distribuição desigual entre os 4 polos** (159 termos em A vs. 1.024-1.848 nos
    outros três) é um resultado genuíno do método, não um erro — mas significa que o
    Vocabulário A é, com os dados disponíveis, um polo estatisticamente mais fraco e
    menos coerente tematicamente que B, C e D (visível na heterogeneidade de seus termos
    na Seção 8). Ler os quatro polos como igualmente robustos seria uma imprecisão.
11. **Regra de fallback para termos de cauda** (2.951 dos 4.354 termos distintivos, Seção
    5, passo 5) atribui um polo com base no documento onde o termo é mais frequente, não
    em uma carga própria calculada — uma classificação estruturalmente mais fraca que a
    dos 1.403 termos com carga direta. A massa agregada dessa cauda é pequena por
    definição (termos de frequência total 1-2), mas o notebook não quantificou
    separadamente quanto dessa cauda pesa em cada categoria por documento — uma lacuna
    que uma iteração futura poderia fechar.

### Sugestões concretas para uma próxima iteração

- Adicionar o plano nacional indiano assim que a Skill01 gerar o `.json` correspondente,
  e reexecutar o notebook — o método (Seções 1-9) já está pronto para 7 documentos sem
  alteração estrutural.
- Testar `df≥3` e `df≥5` como limiares alternativos de "consensual" e comparar a
  estabilidade da composição por categoria (Seção 6) entre os três limiares.
- Aplicar lematização leve (ex.: um lematizador baseado em regras do inglês) e comparar
  se o Vocabulário D permanece dominado por `intelligent`/variantes após a fusão.
- Separar estatisticamente o efeito de formato (itens numerados/estrutura repetitiva) do
  efeito de conteúdo temático — por exemplo, recalculando a composição por categoria após
  remover cabeçalhos estruturais repetidos identificados manualmente.
- Reexecutar a classificação por polos (Seção 5) usando TF-IDF em vez de frequência
  relativa pura como entrada da PCA, e comparar se a orientação dos quatro polos se
  mantém — o mesmo tipo de checagem de robustez já feito com a similaridade de cosseno em
  `analise_geral.ipynb`.

### Perguntas em aberto para orientar a leitura deste notebook

- O forte peso de `intelligent`/`intelligentization` no Vocabulário D é, sobretudo,
  conteúdo (uma escolha real de enquadramento tecnológico chinês) ou, ao menos em parte,
  artefato de tradução? Este notebook não tem como responder sozinho.
- Faz sentido, para a pesquisa em curso, tratar o Vocabulário A como um quarto polo
  "verdadeiro" — dado seu tamanho e heterogeneidade muito menores que B, C e D — ou é
  mais honesto descrevê-lo como um resíduo estatístico dos outros três eixos mais fortes?
- A concentração atípica do `America's AI Action Plan` no Vocabulário C é, sobretudo,
  conteúdo (ênfase real em segurança nacional/estrutura federal) ou efeito de formato
  (itens de "Recommended Policy Action" repetidos)? Separar as duas explicações exigiria
  uma leitura qualitativa do texto completo do documento, fora do escopo deste notebook.

## 12. Uma segunda bússola: vocabulário dirigido por dicionário temático

As Seções 5 a 10 constroem os polos A/B/C/D **de baixo para cima** (*bottom-up*): o
método (PCA sobre o vocabulário distintivo) não sabe, a priori, o que "Vocabulário A"
ou "Vocabulário C" significam — os quatro polos emergem puramente da estrutura
estatística dos dados, e só depois de calculados é que o notebook interpreta, por
leitura dos termos de maior carga (Seção 8), qual tema cada um parece capturar. É uma
escolha deliberada (evita impor uma categoria a priori aos dados), mas tem um custo já
registrado na Seção 11: o polo A, por exemplo, é "o polo mais fraco e mais heterogêneo
dos quatro" (159 termos, sem coerência temática nítida) — o método não garante que os
quatro polos estatísticos correspondam a quatro conceitos igualmente interpretáveis, nem
que o eixo que os separa tenha um rótulo substantivo único e estável.

Esta segunda bússola inverte o caminho — **de cima para baixo** (*top-down*): define,
antes de olhar para qualquer frequência, quatro dicionários temáticos com significado
fixado por leitura de política pública de IA, e só depois mede quanto de cada documento
pertence a cada dicionário. Os rótulos A/B/C/D são reaproveitados de propósito — não
porque seja o "mesmo" Vocabulário A da Seção 5 (não é: a lista de termos é inteiramente
diferente e construída por um critério diferente), mas porque responde à mesma pergunta
de pesquisa (quatro polos de vocabulário distintivo) por um método complementar, com a
vantagem de que o significado de cada polo é conhecido **antes** de qualquer gráfico ser
desenhado, não inferido depois dele. Isso troca a vantagem da PCA (nenhum viés do
analista na escolha dos termos) pela vantagem do dicionário curado (interpretabilidade
garantida por construção) — as duas bússolas deste notebook devem ser lidas como
respostas complementares, não concorrentes, à mesma pergunta.

Os quatro dicionários:

- **Vocabulário A — Desenvolvimento, Inovação e Aceleração.** O lado técnico/produtivo
  do desenvolvimento de IA: pesquisa e desenvolvimento, infraestrutura de computação,
  talento, eficiência, automação, adoção, capacidade de hardware/software, ecossistemas
  de inovação.
- **Vocabulário B — Governança Estatal.** As estruturas institucionais do Estado que
  exercem essa governança: agências, ministérios, comissões, conselhos, órgãos públicos,
  administração pública — quem aplica a política, não a política em si.
- **Vocabulário C — Precaução, Risco e Segurança.** O lado normativo/protetivo: risco,
  segurança, ética, direito, soberania, responsabilidade, transparência, viés,
  mitigação, supervisão.
- **Vocabulário D — Mercado e Competitividade.** O lado econômico/geopolítico: mercado,
  setor privado, empresas, concorrência, comércio exterior, aliados, rivais, cadeia de
  suprimentos.

Os quatro dicionários são aplicados de forma **idêntica** aos seis documentos (mesma
lista, mesmo critério, sem ajuste por documento) sobre a mesma base de vocabulário de
conteúdo pós-exclusão de autoidentificadores já construída na Seção 3
(`final_counts`/`rel_freq`) — nenhuma etapa de tokenização, bigramas ou exclusão é
refeita.

## 13. Os quatro dicionários temáticos — definição e critérios

Cada dicionário foi redigido **antes** de consultar `vocab_union` (a lista de termos
realmente presentes no corpus) — uma lista de candidatos por leitura de domínio (política
pública de IA em inglês), depois cruzada com o corpus para reter apenas os candidatos
que de fato ocorrem em pelo menos um dos seis documentos. Três critérios guiaram a
redação:

1. **Termos genéricos, não nomes próprios.** Nenhum dicionário contém o nome de um país,
   bloco, ministério, agência ou iniciativa específicos — essa é exatamente a classe de
   termo que a Seção 3 já remove de `final_counts`/`rel_freq` antes de qualquer cálculo
   aqui. Termos de fronteira (nomes de leis, de órgãos criados por uma lei específica)
   são tratados caso a caso e documentados na tabela abaixo.
2. **Sem lematização própria.** Como a Seção 1 já declarou, o notebook não lematiza — por
   isso cada dicionário lista explicitamente as variantes morfológicas mais frequentes de
   cada lema (`accelerate`/`accelerating`/`acceleration`/`accelerated`, por exemplo), não
   apenas a forma-base.
3. **Categorias mutuamente exclusivas.** Nenhum termo aparece em mais de um dos quatro
   dicionários (verificado por código, célula seguinte) — cada termo do vocabulário do
   corpus classificado por este método pertence a exatamente um polo, nunca a dois.

**Casos de fronteira, decididos e documentados (não escondidos):**

| Termo(s) | Decisão | Justificativa |
|---|---|---|
| `governance`, `AI governance` | Vocabulário **C**, não B | Nestes documentos, "governança" é usado para o arcabouço de regras/supervisão de risco (transparência, responsabilização, supervisão humana) — o conceito abstrato de regulação, não o aparato que a executa. Mantido separado de `government`/`governmental`/`governing` (Vocabulário B), que referenciam o Estado como ator institucional. |
| `act`, `AI Act` | Vocabulário **C** | É um instrumento legal (lei), não um órgão de governo — tratado como conteúdo temático de "law/act" (nomeado explicitamente como exemplo de Vocabulário C), no mesmo espírito em que a Seção 3 manteve `Horizon Europe` como conteúdo, não autoidentificação. Ressalva registrada na Seção 17: em outra leitura, mais rígida, seria também uma autoidentificação da política regulatória europeia especificamente. |
| `AI Office`, `AI Board` | **Excluídos** de todos os dicionários | São órgãos específicos criados pelo AI Act europeu — tratados como autoidentificação institucional da UE, no mesmo critério que excluiu `State Council` (China) e `NIST`/`CAISI` (EUA) na Seção 3. |
| `Trump administration`, `Horizon Europe`, `Competitiveness Compass` | **Excluídos** | Nomes próprios de administração/programa específicos de um único país/bloco — mesmo critério da Seção 3. |
| `talent`, `AI talent`, `skills`/`AI skills` | Vocabulário **A** | Tratados como insumo para o desenvolvimento de IA (capacidade técnica). |
| `workforce`, `employment`, `jobs`, `job`, `productivity` | Vocabulário **D** | Tratados como consequência econômica/de mercado de trabalho, não como desenvolvimento técnico — uma distinção interpretativa entre "construir a capacidade de fazer IA" (A) e "o efeito econômico de ter IA" (D), não uma verdade objetiva. |
| `partnership`, `partnerships`, `partner`, `partners`, `partnering` | Vocabulário **D** | Nestes documentos, predominantemente cooperação internacional/econômica — agrupado com `allies`/`alliance`. Ressalva: o mesmo token também cobre parcerias de pesquisa (mais próximas de A) e parcerias público-privadas (fronteira com B), sentidos que o token isolado não distingue. |
| `policy`, `policies`, `public` (isolado), `human` (isolado), `civil`, `due`, `AI Alliance` | **Não classificados** em nenhum dicionário | Julgados excessivamente polissêmicos neste corpus para indicar com confiança um único polo (ex.: `public` ocorre tanto em "public sector" — Vocabulário B — quanto em "publicly available" — sentido nenhum dos quatro polos captura). Preferiu-se deixar de fora a classificar errado. |

Essa tabela cumpre, para os dicionários A-D, a mesma função de transparência que a tabela
de autoidentificadores da Seção 3 cumpriu para aquela etapa: nenhuma decisão de inclusão
ou exclusão de termo ambíguo é feita silenciosamente.

In [ ]:
VOCAB_A = [
 "accelerate","accelerating","acceleration","accelerated","accelerator","accelerators",
 "innovation","innovative","innovations","innovate","innovators","innovating",
 "innovation hubs","innovation ecosystem","innovation-driven","innovation-friendly","innovation-oriented",
 "breakthrough","breakthroughs","cutting-edge","frontier","frontier AI",
 "development","develop","developing","developed","developments","developers","developer",
 "deployment","deploy","deployed","deploying",
 "efficiency","efficient","efficiently",
 "automation","automated","automate","automating",
 "adoption","adopt","adopted","adopting",
 "large-scale","scale-up","scaleup","scaleups",
 "modernization","modernize","modernise",
 "transformative","transformation","transform","digital transformation","transformational",
 "hub","hubs","cluster","clusters",
 "potential",
 "catalyst","catalysts","incentive","incentives","incentivize","incentivise","incentivising",
 "streamline","streamlined","streamlining",
 "agility","agile",
 "momentum","pioneer",
 "leading","world-leading",
 "talent","AI talent","talents",
 "computing","compute","computational","computational power",
 "supercomputers","supercomputing","supercomputer","high-performance computing","high-performance processing",
 "infrastructure","AI infrastructure","infrastructures",
 "R&D","Science & Technology",
 "open source","foundation model","generative AI","general-purpose AI","large language (model)","machine learning","natural language",
 "chips","chip","semiconductors","semiconductor","semiconductor manufacturing","hardware","software",
 "robots","robotics","robot","robotic",
 "AI ecosystem","AI skills","AI literacy","AI strategy",
 "data center","data spaces",
 "smart terminal","automatic driving","autonomous unmanned","swarm intelligence","virtual reality","hybrid enhanced (intelligence)",
]

VOCAB_C = [
 "risk","risks","high-risk","risk management","risk-based",
 "safety","safe","safeguards","safeguard","safer","safety-critical",
 "security","secure","national security","cybersecurity","securing","secure-by-design",
 "ethical","ethics","ethically",
 "regulatory","regulations","regulation","regulatory framework","regulated","deregulation",
 "law","laws","lawful",
 "act","AI Act",
 "sovereignty","sovereign",
 "harm","harmful",
 "bias","biases","fair","fairness",
 "transparency","transparent","transparently",
 "trust","trustworthy","trusted","trustworthy AI","trustworthiness",
 "accountability",
 "threats","threat","threaten",
 "vulnerabilities","vulnerability","vulnerable",
 "mitigation","mitigate","mitigating",
 "misused","abuse",
 "surveillance",
 "reliable","reliability","liability",
 "protection","protect","protecting","protected","protects",
 "standards","standard","standardization",
 "critical infrastructure",
 "robust","robustness","resilience","resilient",
 "alignment","align","aligned","aligning","aligns",
 "interpretability","interpretable","explainable",
 "certification","certified",
 "incident","incidents",
 "proliferation",
 "enforcement","testing","evaluation","evaluations","auditing",
 "responsible AI","sustainable AI",
 "rights",
 "governance","AI governance",
]

VOCAB_B = [
 "agencies","agency","interagency",
 "ministry","ministries",
 "administration","administrations","administrative",
 "interministerial",
 "department","departments",
 "commission","commissions",
 "council","councils",
 "bureau","bureaucracy","bureaucratic",
 "authorities","authority","regulators",
 "member states",
 "public sector","public consultation",
 "office","offices","officer","officers","officials","official",
 "board","boards","committee","committees","subcommittee",
 "federal","state-owned",
 "state","states",
 "government","governmental","governments","govern","governing",
 "stakeholders","stakeholder","participation","consultation",
]

VOCAB_D = [
 "market","markets","single market","labor market","market-dominant","market-leading","marketplace",
 "private sector","private","private-sector",
 "competitiveness","competitive","competition","competitions","competitors","competing","compete",
 "trade","export","exports","exporting","export controls",
 "allies","ally","alliance","alliances",
 "adversaries","adversary","adversarial","rivals","geopolitical","geopolitics","race",
 "corporation","corporations","corporate",
 "company","companies",
 "industry","industries","industrial","industry-specific","industrial chain",
 "business","businesses",
 "investment","investments","invest","investing","investors",
 "capital","capitalising","capitalize",
 "entrepreneurship","entrepreneurs","entrepreneurial",
 "startup","startups","venture","ventures",
 "workforce","employment","employers","employees","jobs","job",
 "productivity","productive","growth",
 "revenue","gdp",
 "supply chain","value chain",
 "intellectual property","patents","patent","copyright",
 "smes","tariffs",
 "dominance","dominant",
 "partnership","partnerships","partner","partners","partnering",
]

DICTS = {"A": VOCAB_A, "B": VOCAB_B, "C": VOCAB_C, "D": VOCAB_D}
print("Tamanho dos dicionários-candidato (antes de cruzar com o corpus):",
      {cat: len(lst) for cat, lst in DICTS.items()}, "total:", sum(len(v) for v in DICTS.values()))

In [ ]:
# Checagem 1: nenhum termo em mais de um dicionário (categorias mutuamente exclusivas)
seen = {}
duplicates = []
for cat, lst in DICTS.items():
    for t in lst:
        if t in seen:
            duplicates.append((t, seen[t], cat))
        seen[t] = cat
assert not duplicates, f"termos duplicados entre dicionários: {duplicates}"
print("Checagem de exclusividade mútua: OK — nenhum termo aparece em mais de um dicionário.\n")

# Checagem 2: quais candidatos de fato ocorrem no corpus (vocab_union, pós Seção 3)
matched, unmatched = {}, {}
for cat, lst in DICTS.items():
    matched[cat] = sorted(set(lst) & vocab_union)
    unmatched[cat] = sorted(set(lst) - vocab_union)
    print(f"Vocabulário {cat}: {len(lst):3d} candidatos -> {len(matched[cat]):3d} encontrados no corpus "
          f"({len(unmatched[cat])} não encontrados: {unmatched[cat]})")

total_candidatos = sum(len(v) for v in DICTS.values())
total_encontrados = sum(len(v) for v in matched.values())
print(f"\nTotal: {total_candidatos} candidatos, {total_encontrados} encontrados em pelo menos um dos "
      f"seis documentos ({total_encontrados/total_candidatos*100:.1f}%).")

Dos 357 termos-candidato definidos a priori nos quatro dicionários, 355 (99,4%) ocorrem
de fato em pelo menos um dos seis documentos — apenas as formas plurais `committees` e
`exports` não aparecem (as formas singulares `committee` e `export` já estão presentes e
cobrem o mesmo conceito). Isso indica que os dicionários foram redigidos com vocabulário
realista para este domínio, não uma lista genérica desalinhada do corpus.

## 14. Composição de cada documento nos quatro dicionários temáticos

Para cada documento, soma-se a frequência relativa (`rel_freq`, a mesma base normalizada
da Seção 4 em diante) de todos os termos que casam com cada um dos quatro dicionários.
**Diferença metodológica importante em relação à Seção 6:** ali, a partição
Consensual/A/B/C/D era **exaustiva** (todo termo do vocabulário de conteúdo pertencia a
exatamente uma categoria, soma = 100%). Aqui, a classificação **não é exaustiva** — um
termo só entra em A, B, C ou D se estiver literalmente em um dos quatro dicionários
curados; a maior parte do vocabulário de qualquer documento (conectivos temáticos
genéricos, vocabulário específico de domínio não coberto pelos quatro eixos, termos
excluídos por ambiguidade na Seção 13) fica fora dos quatro polos. Por isso a tabela
abaixo reporta explicitamente a **cobertura** (A+B+C+D) de cada documento — a fração do
seu vocabulário que esta bússola de fato classifica — em vez de assumir que ela é 100%.

In [ ]:
mass2 = {k: {cat: sum(rel_freq[k].get(t, 0.0) for t in matched[cat]) for cat in DICTS} for k in doc_keys}

comp_df2 = pd.DataFrame({
    "Documento": [f"{docs[k]['label']} ({docs[k]['pais_ou_bloco']})" for k in doc_keys],
    **{f"{cat} (%)": [round(mass2[k][cat]*100, 2) for k in doc_keys] for cat in ["A", "B", "C", "D"]},
})
comp_df2["Cobertura A+B+C+D (%)"] = comp_df2[[f"{c} (%)" for c in "ABCD"]].sum(axis=1).round(2)
comp_df2["Não classificado (%)"] = (100 - comp_df2["Cobertura A+B+C+D (%)"]).round(2)
comp_df2

In [ ]:
# Checagem de robustez: nenhum polo, em nenhum documento, deve ser dominado por um único termo isolado
worst = []
for cat in DICTS:
    for k in doc_keys:
        contribs = sorted([rel_freq[k].get(t, 0.0) for t in matched[cat]], reverse=True)
        if mass2[k][cat] > 0:
            worst.append((contribs[0] / mass2[k][cat], cat, k))
worst.sort(reverse=True)
top_frac, top_cat, top_k = worst[0]
print(f"Maior concentração observada em um único termo: {top_frac*100:.1f}% da massa do Vocabulário "
      f"{top_cat} em '{docs[top_k]['label']}'.")
print("Nenhuma concentração se aproxima do caso extremo da Seção 10 (\"intelligent\", >2x o segundo colocado) —")
print("a massa de cada polo, em cada documento, está distribuída entre vários termos, não dominada por um só.")

**Leitura da tabela.** A cobertura dos quatro dicionários varia entre **13,5%**
(`New Generation AI Development Plan`) e **19,9%** (`America's AI Action Plan`), média de
15,7% — ou seja, mesmo no documento mais bem coberto, cerca de **80%** do vocabulário de
conteúdo permanece fora dos quatro polos temáticos. Isso não é uma falha do método: é a
consequência esperada de usar dicionários curados e conceitualmente estritos (Seção 13)
em vez de uma partição estatística exaustiva (Seção 6) — a bússola desta seção classifica
com confiança uma fatia menor do vocabulário, mas cada termo que ela classifica tem
significado temático conhecido a priori, o que a bússola da Seção 6 não garante. A
checagem de concentração confirma que nenhum polo, em nenhum documento, é um artefato de
um único termo hiperfrequente — ao contrário do que a Seção 10 registrou para o termo
`intelligent` na bússola estatística.

## 15. Dois eixos objetivos: Estado↔Mercado e Precaução↔Desenvolvimento

A posição de cada documento na nova bússola é definida por **duas diferenças de massa
relativa**, calculadas diretamente a partir da tabela da Seção 14 — nenhuma projeção
estatística (PCA/SVD) está envolvida nesta segunda bússola, ao contrário da primeira:

- **Eixo X = massa relativa do Vocabulário D − massa relativa do Vocabulário B**, em
  pontos percentuais do vocabulário do documento. Positivo desloca o documento para o
  lado de "Mercado e Competitividade"; negativo, para o lado de "Governança Estatal".
- **Eixo Y = massa relativa do Vocabulário A − massa relativa do Vocabulário C**, também
  em pontos percentuais. Positivo desloca o documento para "Desenvolvimento/Aceleração";
  negativo, para "Precaução/Risco".

Esta é uma métrica deliberadamente simples e auditável — qualquer leitor com a tabela da
Seção 14 em mãos pode recalcular os dois eixos com uma subtração, sem depender de
autovalores, sinais arbitrários de componente ou qualquer transformação não linear
(comparar com a Seção 5, onde o próprio notebook alerta que "o sinal de PC1 e de PC2 é
matematicamente arbitrário"). O preço dessa simplicidade é a escala: como a Seção 14
mostra, os Vocabulários A e C têm massa agregada tipicamente maior que B e D neste
corpus — logo o eixo Y tende a ter maior amplitude entre documentos do que o eixo X. Isso
é reportado, não corrigido por reescalonamento artificial (ver Seção 17).

In [ ]:
eixo_x = {k: (mass2[k]["D"] - mass2[k]["B"]) * 100 for k in doc_keys}
eixo_y = {k: (mass2[k]["A"] - mass2[k]["C"]) * 100 for k in doc_keys}
cobertura2 = {k: sum(mass2[k].values()) * 100 for k in doc_keys}

pos_df2 = pd.DataFrame({
    "Documento": [docs[k]["label"] for k in doc_keys],
    "país/bloco": [docs[k]["pais_ou_bloco"] for k in doc_keys],
    "Eixo X = D-B (pp)": [round(eixo_x[k], 2) for k in doc_keys],
    "Eixo Y = A-C (pp)": [round(eixo_y[k], 2) for k in doc_keys],
    "Cobertura A+B+C+D (%)": [round(cobertura2[k], 2) for k in doc_keys],
})
pos_df2

**Achado que antecede o gráfico.** Nos seis documentos, sem exceção, o Eixo Y é
positivo — todos pendem para "Desenvolvimento" mais do que para "Precaução" em termos de
volume de vocabulário, o que é esperado (são planos nacionais de *promoção* de IA, não
relatórios de auditoria de risco) e é, em si, um resultado substantivo: a bússola
dirigida por dicionário não encontrou nenhum dos seis documentos mais próximo do polo de
precaução do que do polo de desenvolvimento. O Eixo X é mais disputado — cinco documentos
têm X levemente positivo (mais Mercado do que Estado), e o `America's AI Action Plan` é o
único com X marginalmente negativo (-0,08pp, praticamente no eixo). A leitura completa,
documento a documento, está na Seção 16, após o gráfico.

## 16. A bússola dirigida por dicionário — visualização

Ao contrário das Seções 7-8 (eixos sem unidade, apenas ordinais, com "caixas de polo" nos
quatro extremos), esta bússola usa eixos numéricos contínuos em pontos percentuais — o
leitor pode ler o valor exato de cada posição diretamente da régua, não apenas seu
quadrante. O tamanho de cada bolha replica a convenção da Seção 7 (quanto maior, mais
vocabulário do documento os quatro dicionários conseguem classificar — a "cobertura" da
Seção 14), e a cor identifica o país/bloco, com a mesma paleta `bloc_color` já definida na
Seção 7.

In [ ]:
LABEL_OFFSET2 = {
    "pbia":            (-0.14, -0.10, "right",  "center"),
    "ai_plus":         (-0.14,  0.34, "right",  "bottom"),
    "new_generation":  ( 0.00,  0.40, "center", "bottom"),
    "americas":        ( 0.00, -0.42, "center", "top"),
    "apply_ai":        ( 0.00, -0.42, "center", "top"),
    "ai_continent":    ( 0.16,  0.30, "left",   "bottom"),
}

xs = np.array([eixo_x[k] for k in doc_keys])
ys = np.array([eixo_y[k] for k in doc_keys])
xlim = max(abs(xs).max(), 1.0) * 1.45
ylim = max(abs(ys).max(), 1.0) * 1.18

fig, ax = plt.subplots(figsize=(10.5, 8), dpi=130)
fig.patch.set_facecolor(SURFACE); ax.set_facecolor(SURFACE)

ax.axvspan(0, xlim, ymin=0.5, ymax=1, color=COL_YELLOW, alpha=0.06, zorder=0)
ax.axvspan(-xlim, 0, ymin=0.5, ymax=1, color=COL_AQUA, alpha=0.06, zorder=0)
ax.axvspan(0, xlim, ymin=0, ymax=0.5, color=COL_RED, alpha=0.06, zorder=0)
ax.axvspan(-xlim, 0, ymin=0, ymax=0.5, color=COL_BLUE, alpha=0.06, zorder=0)
ax.axhline(0, color=AXIS, lw=1.3, zorder=2)
ax.axvline(0, color=AXIS, lw=1.3, zorder=2)

corner_kw = dict(fontsize=8.2, fontweight="bold", color=INK_MUTED, alpha=0.8, zorder=3)
ax.text(xlim*1.10, -ylim*0.94, "MERCADO + PRECAUÇÃO (D + C)", ha="right", va="bottom", **corner_kw)
ax.text(-xlim*0.96, -ylim*0.94, "ESTADO + PRECAUÇÃO (B + C)", ha="left", va="bottom", **corner_kw)

sizes = np.array([cobertura2[k] for k in doc_keys])
for i, k in enumerate(doc_keys):
    bloc = docs[k]["pais_ou_bloco"]; color = bloc_color[bloc]
    ax.scatter(eixo_x[k], eixo_y[k], s=sizes[i]*38, color=color, alpha=0.55, edgecolor=color, linewidth=1.8, zorder=4)
    ax.scatter(eixo_x[k], eixo_y[k], s=22, color=color, zorder=5)
    dx, dy, ha, va = LABEL_OFFSET2[k]
    ax.annotate(f"{doc_label[k]} ({docs[k]['pais_ou_bloco']})\nX={eixo_x[k]:+.2f}pp · Y={eixo_y[k]:+.2f}pp",
                xy=(eixo_x[k], eixo_y[k]), xytext=(eixo_x[k]+dx, eixo_y[k]+dy),
                fontsize=7.6, color=INK_PRIMARY, ha=ha, va=va, linespacing=1.5, zorder=6)

ax.set_xlim(-xlim, xlim*1.12); ax.set_ylim(-ylim, ylim)
ax.set_xlabel("← Vocabulário B (governança estatal)      Vocabulário D (mercado/competitividade) →\n"
              "(pontos percentuais: massa relativa D − massa relativa B)", fontsize=8.6, color=INK_SECONDARY)
ax.set_ylabel("← Vocabulário C (precaução/risco)      Vocabulário A (desenvolvimento/aceleração) →\n"
              "(pontos percentuais: massa relativa A − massa relativa C)", fontsize=8.6, color=INK_SECONDARY)
for s in ["top", "right"]: ax.spines[s].set_visible(False)
for s in ["left", "bottom"]: ax.spines[s].set_color(AXIS)
ax.tick_params(colors=INK_MUTED, labelsize=8)
ax.set_title("Bússola temática dirigida por dicionário — posição de cada documento\n"
             "tamanho da bolha = % do vocabulário do documento classificado em A, B, C ou D (cobertura dos dicionários)",
             fontsize=11.5, color=INK_PRIMARY, pad=14, fontweight="bold")
plt.tight_layout()
plt.show()

**Leitura, documento a documento:**

- **Os dois documentos chineses ocupam o canto mais "Desenvolvimento" do mapa** — a
  `New Generation AI Development Plan` é o ponto mais extremo do eixo Y de todo o corpus
  (+6,70pp), seguida pela `"AI+" Initiative` (+4,87pp); ambas também levemente positivas
  em X. Coerente com a leitura já feita na bússola estatística (Seção 7): os dois planos
  chineses são, pelas duas metodologias independentes, os mais concentrados em vocabulário
  de desenvolvimento/aceleração e os menos concentrados em vocabulário de precaução.
- **Os dois documentos europeus ficam entre os chineses e o americano**, próximos entre
  si em Y (`AI Continent Action Plan` +5,19pp, `Apply AI Strategy` +3,60pp) e com X
  moderadamente positivo — puxado majoritariamente pelo termo `AI Act` no Vocabulário C
  (Seção 13) e por `member states`/`public sector` no Vocabulário B (Seção 14):
  mesmo sendo os documentos com a redação regulatória mais explícita do corpus (o próprio
  `AI Act` é uma lei), o volume de vocabulário de "Mercado" (`competitiveness`,
  `single market`, `companies`) supera o de "Governança Estatal" nestes dois documentos.
- **O `America's AI Action Plan` é o único ponto do lado esquerdo do eixo Y=0 em X**
  (-0,08pp) e tem o menor Y de todo o corpus (+1,47pp) — o documento mais equilibrado
  entre os quatro vocabulários, e o de maior cobertura (19,9%, a maior bolha do gráfico).
  Isso não decorre de um tom mais "precaucionista": a Seção 14 mostra que o `America's AI
  Action Plan` tem, em termos absolutos, a **maior** massa em Vocabulário C de todo o
  corpus (4,81%) — mas o termo que mais contribui para essa massa é `national security`
  (não `safety`/`ethics`), e a mesma leitura vale para o Vocabulário B: sua massa (4,44%)
  é dominada por `federal`/`agencies`/`government`, um efeito da estrutura do documento
  (itens de "Recommended Policy Action" endereçados a agências federais específicas, já
  registrado como limitação na Seção 11) mais do que uma ênfase estatal deliberada.
- **O PBIA fica em posição intermediária** (X=+0,86pp, Y=+3,53pp) — mais próximo do
  centro do que qualquer documento chinês ou europeu, mas do lado "Desenvolvimento" e
  "Mercado", nunca cruzando para "Precaução" ou "Estado".

**O que o gráfico não sustenta:** nenhum dos seis documentos aparece no semiplano
inferior (Y negativo, "Precaução/Risco" dominante) — os dois quadrantes inferiores
(sombreados na figura) estão vazios. Não seria correto ler essa ausência como "nenhum
destes documentos discute risco" (a Seção 14 mostra massa C > 0 em todos); o que a
bússola mostra é que, nos seis documentos, o volume relativo de vocabulário de
desenvolvimento sempre supera o de precaução — uma característica do gênero textual
"plano nacional de promoção de IA", não necessariamente do conteúdo de risco em si
(discutido com mais detalhe na Seção 17).

## 17. Metodologia, análise e limitações da bússola dirigida por dicionário

### O que foi feito

Esta segunda bússola parte da mesma base de dados pós-limpeza da Seção 3
(`final_counts`/`rel_freq`, seis documentos, autoidentificadores nacionais/institucionais
já excluídos) e substitui o método estatístico das Seções 5-10 (PCA sobre o vocabulário
distintivo, polos emergentes) por um método dedutivo: (1) quatro dicionários temáticos —
Desenvolvimento/Inovação/Aceleração (A), Governança Estatal (B), Precaução/Risco/Segurança
(C) e Mercado/Competitividade (D) — foram redigidos por leitura de domínio, antes de
qualquer consulta ao corpus (Seção 13); (2) os candidatos foram cruzados com o vocabulário
realmente presente nos seis documentos, mantendo apenas os termos que ocorrem (355 de 357
candidatos, 99,4%); (3) verificou-se por código que os quatro dicionários são mutuamente
exclusivos (nenhum termo em mais de um) e que nenhum polo, em nenhum documento, é
dominado por um único termo isolado (máximo observado: 35,3%); (4) para cada documento,
somou-se a frequência relativa dos termos de cada dicionário, produzindo quatro massas
(A, B, C, D) e uma medida explícita de cobertura (quanto do vocabulário do documento os
quatro dicionários conseguem classificar — entre 13,5% e 19,9%, Seção 14); (5) dois eixos
objetivos e diretamente auditáveis foram definidos como diferenças de massa em pontos
percentuais — Eixo X = D−B (Mercado menos Estado), Eixo Y = A−C (Desenvolvimento menos
Precaução) — e usados para posicionar cada documento em um plano cartesiano com escala
numérica explícita (Seção 15-16).

### Metodologia em relação à primeira bússola (Seções 5-10)

| | Bússola estatística (Seções 5-10) | Bússola dirigida por dicionário (Seções 12-17) |
|---|---|---|
| Direção | Bottom-up (polos emergem dos dados) | Top-down (polos definidos antes dos dados) |
| Base de termos | Todo o vocabulário distintivo (df≤3), 4.354 termos | 4 dicionários curados, 355 termos encontrados no corpus |
| Eixos | PC1/PC2 de uma SVD (sinal arbitrário) | Diferenças de massa relativa (unidade fixa: pontos percentuais) |
| Cobertura do vocabulário | Exaustiva (partição em 5 categorias, soma sempre 100%) | Parcial e declarada (13,5%–19,9%, resto não classificado) |
| Interpretação dos polos | Inferida depois, por leitura dos termos de maior carga | Fixada antes, por definição do dicionário |
| Risco principal | Um termo isolado pode dominar um polo (caso `intelligent`, Seção 10) | Um termo ambíguo classificado errado distorce o polo inteiro (mitigado pela tabela de casos de fronteira, Seção 13) |

As duas bússolas concordam em um ponto qualitativo central — os dois documentos chineses
mais "desenvolvimento/aceleração" e mais distantes do polo de precaução, os dois europeus
marcados por vocabulário regulatório/institucional, o `America's AI Action Plan` como o
documento mais atípico do corpus — o que fortalece a confiança nesse padrão específico,
por ter sido obtido por dois métodos independentes. Elas divergem, porém, na posição
relativa do PBIA e na magnitude da separação entre EUA e os demais, o que é esperado:
métodos diferentes, aplicados à mesma pergunta, não precisam concordar em todos os
detalhes para serem ambos válidos.

### Limitações específicas desta bússola

1. **Os dicionários são uma escolha do analista, não um fato do corpus.** Cada termo
   incluído ou excluído (Seção 13) reflete uma leitura interpretativa de política pública
   de IA em inglês — outro leitor, com outra lista, obteria pontos diferentes. A tabela de
   casos de fronteira torna essas escolhas auditáveis, não as torna objetivamente
   corretas.
2. **Cobertura baixa e desigual (13,5%–19,9%).** A posição de cada documento resume, no
   melhor caso, a quinta parte do seu vocabulário de conteúdo — os outros quatro quintos
   (vocabulário consensual da área, termos idiossincráticos fora dos quatro eixos) não
   têm influência sobre X e Y. Uma cobertura maior exigiria dicionários mais permissivos,
   à custa de introduzir mais termos ambíguos (o trade-off documentado na Seção 13).
3. **Amplitude desigual entre os dois eixos.** Como a Seção 15 antecipou, o eixo Y
   (Desenvolvimento−Precaução) tem amplitude maior entre os seis documentos do que o eixo
   X (Mercado−Estado) — não porque a distinção Estado/Mercado seja menos real neste
   corpus, mas porque os Vocabulários A e C, tal como definidos, capturam mais massa
   agregada de vocabulário do que B e D (comparar tamanhos dos dicionários e suas taxas de
   ocorrência na Seção 14). Nenhum reescalonamento (p.ex., padronização por eixo) foi
   aplicado para corrigir essa assimetria — os valores em pontos percentuais são reportados
   como estão, por serem diretamente interpretáveis; um reescalonamento tornaria os eixos
   mais "equilibrados" visualmente, mas às custas de trocar uma unidade concreta (pontos
   percentuais de vocabulário) por uma unidade relativa ao próprio conjunto de seis
   documentos (desvios-padrão), menos direta de explicar a um leitor.
4. **O Vocabulário C mistura duas famílias de precaução que nem sempre coincidem.**
   Termos como `national security`, `cybersecurity` e `adversarial` (segurança
   estratégica/geopolítica) e termos como `ethics`, `fairness` e `human oversight`
   (segurança/ética do sistema de IA em si) foram deliberadamente agrupados em um único
   polo C. Isso é definido assim porque ambos respondem à mesma pergunta de pesquisa
   ("o documento enquadra a IA como algo a conter/precaver?"), mas são conceitualmente
   distintos — e a Seção 16 mostra um caso concreto em que essa mistura afeta a leitura:
   a massa C do `America's AI Action Plan` é alta principalmente por `national security`,
   não por vocabulário de ética/segurança do sistema, o que um leitor apressado do gráfico
   poderia confundir.
5. **N=6 documentos, mesma limitação estrutural da Seção 11.** Qualquer leitura como
   padrão geral de "documentos chineses" ou "documentos europeus" de política de IA seria
   uma generalização indevida — o achado é sobre estes seis textos.
6. **Tradução institucional como fator não controlado**, idêntico ao ponto 2 da Seção 11:
   os dois documentos chineses e o PBIA chegam ao corpus via tradução institucional ou
   redação já em inglês não nativo — o vocabulário que alimenta os quatro dicionários pode
   refletir, em parte, convenção de tradução, não apenas escolha de conteúdo.
7. **Sensibilidade não testada.** O notebook não testou quanto a posição de cada
   documento mudaria sob dicionários alternativos (por exemplo, movendo `partnership`
   para "não classificado" em vez de D, ou `AI Act` para "excluído" em vez de C) — os
   casos de fronteira da Seção 13 identificam onde a robustez do resultado é mais
   incerta, mas não quantificam o efeito de cada escolha alternativa.

### Sugestões para uma próxima iteração

- Testar a sensibilidade do resultado a variações nos dicionários (remover/adicionar os
  termos de fronteira da Seção 13, um de cada vez) e reportar o quanto X e Y de cada
  documento se deslocam.
- Separar o Vocabulário C em dois sub-eixos (segurança estratégica/geopolítica vs.
  ética/segurança do sistema de IA) e verificar se a posição do `America's AI Action Plan`
  se mantém distinta nos dois.
- Adicionar o plano nacional indiano assim que disponível (mesma limitação da Seção 11) e
  recalcular as duas bússolas para comparação direta em sete documentos.
- Repetir o exercício com dicionários traduzidos e aplicados também aos trechos em
  português do PBIA anteriores à tradução para inglês usada neste corpus, para isolar o
  efeito de tradução mencionado na limitação 6.